## 1. Imports and configuration

In [ ]:
from __future__ import annotations

import gc
import os
import subprocess
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")

TARGET = "addicted_label"
ID_COL = "id"

NUM_COLS = [
    "age",
    "daily_screen_time_hours",
    "social_media_hours",
    "gaming_hours",
    "work_study_hours",
    "sleep_hours",
    "notifications_per_day",
    "app_opens_per_day",
    "weekend_screen_time",
]
CAT_COLS = ["gender", "stress_level", "academic_work_impact"]
RAW_COLS = NUM_COLS + CAT_COLS

OUTER_FOLDS = 5
INNER_FOLDS = 5
SEED = 20260826
TE_SMOOTH = 10.0
MISSING_SENTINEL = -1_000_000.0


## 2. Data paths, validation, and fold-safe encoding

In [ ]:
class Paths:
    train: str
    test: str
    sample: str
    output: str = "/kaggle/working/submission_s6e8_frontier.csv"
    oof: str = "/kaggle/working/oof_s6e8_frontier.csv"


def locate_competition_files() -> Paths:
    candidates = [
        "/kaggle/input/competitions/playground-series-s6e8",
        "/kaggle/input/playground-series-s6e8",
        ".",
    ]
    for root in candidates:
        tr = os.path.join(root, "train.csv")
        te = os.path.join(root, "test.csv")
        ss = os.path.join(root, "sample_submission.csv")
        if all(os.path.exists(p) for p in (tr, te, ss)):
            return Paths(tr, te, ss)

    # Final fallback: search Kaggle input tree.
    found = {}
    if os.path.exists("/kaggle/input"):
        for root, _, files in os.walk("/kaggle/input"):
            for name in ("train.csv", "test.csv", "sample_submission.csv"):
                if name in files and name not in found:
                    found[name] = os.path.join(root, name)
            if len(found) == 3:
                break

    if len(found) == 3:
        return Paths(found["train.csv"], found["test.csv"], found["sample_submission.csv"])

    raise FileNotFoundError(
        "Could not find train.csv, test.csv and sample_submission.csv. "
        "Attach the S6E8 competition data to this Kaggle notebook."
    )


def has_gpu() -> bool:
    try:
        return subprocess.run(
            ["nvidia-smi"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            timeout=5,
        ).returncode == 0
    except Exception:
        return False


def validate_raw(train: pd.DataFrame, test: pd.DataFrame, sample: pd.DataFrame) -> None:
    exp_train = {ID_COL, TARGET, *RAW_COLS}
    exp_test = {ID_COL, *RAW_COLS}
    if set(train.columns) != exp_train:
        raise ValueError(f"Unexpected train columns: {list(train.columns)}")
    if set(test.columns) != exp_test:
        raise ValueError(f"Unexpected test columns: {list(test.columns)}")
    if list(sample.columns) != [ID_COL, TARGET]:
        raise ValueError(f"Unexpected sample columns: {list(sample.columns)}")
    if len(test) != len(sample):
        raise ValueError("test.csv and sample_submission.csv have different row counts")
    if not test[ID_COL].reset_index(drop=True).equals(sample[ID_COL].reset_index(drop=True)):
        raise ValueError("test IDs do not exactly match sample_submission IDs")
    if train[ID_COL].duplicated().any() or test[ID_COL].duplicated().any():
        raise ValueError("Duplicate IDs detected")
    if train[TARGET].isna().any() or not set(train[TARGET].unique()).issubset({0, 1}):
        raise ValueError("Target must be non-missing binary 0/1")


def prep_categories(train: pd.DataFrame, test: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    train = train.copy()
    test = test.copy()
    for c in CAT_COLS:
        # Same category universe without using labels.
        cats = pd.Index(
            pd.concat([train[c], test[c]], ignore_index=True)
            .dropna()
            .astype(str)
            .unique()
        )
        train[c] = pd.Categorical(train[c], categories=cats)
        test[c] = pd.Categorical(test[c], categories=cats)
    return train, test


def _smoothed_map(values: pd.Series, y: np.ndarray, smoothing: float) -> pd.Series:
    prior = float(np.mean(y))
    keys = values.astype("float64").fillna(MISSING_SENTINEL)
    frame = pd.DataFrame({"key": keys.to_numpy(), "y": y})
    stats = frame.groupby("key", sort=False)["y"].agg(["sum", "count"])
    return (stats["sum"] + smoothing * prior) / (stats["count"] + smoothing)


def _apply_map(values: pd.Series, mapping: pd.Series, fallback: float) -> np.ndarray:
    keys = values.astype("float64").fillna(MISSING_SENTINEL)
    return keys.map(mapping).fillna(fallback).to_numpy(dtype=np.float32)


def nested_exact_value_te(
    tr: pd.DataFrame,
    y: np.ndarray,
    va: pd.DataFrame,
    te: pd.DataFrame,
    seed: int,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Fold-safe exact-value TE.

    Training encodings are generated by an INNER CV, so each row's target is
    excluded from the mapping used for that row. Validation/test mappings use
    only the outer training fold.
    """
    prior = float(np.mean(y))
    enc_tr = pd.DataFrame(index=np.arange(len(tr)))
    enc_va = pd.DataFrame(index=np.arange(len(va)))
    enc_te = pd.DataFrame(index=np.arange(len(te)))

    inner = StratifiedKFold(n_splits=INNER_FOLDS, shuffle=True, random_state=seed)
    inner_splits = list(inner.split(np.zeros(len(y)), y))

    for col in NUM_COLS:
        oof_col = np.empty(len(tr), dtype=np.float32)
        for fit_idx, hold_idx in inner_splits:
            mapping = _smoothed_map(tr.iloc[fit_idx][col], y[fit_idx], TE_SMOOTH)
            oof_col[hold_idx] = _apply_map(tr.iloc[hold_idx][col], mapping, prior)

        full_map = _smoothed_map(tr[col], y, TE_SMOOTH)
        name = f"te__{col}"
        enc_tr[name] = oof_col
        enc_va[name] = _apply_map(va[col], full_map, prior)
        enc_te[name] = _apply_map(te[col], full_map, prior)

    return enc_tr, enc_va, enc_te


## 3. Feature engineering and ensemble utilities

In [ ]:
def add_features(raw: pd.DataFrame, encoded: pd.DataFrame | None) -> pd.DataFrame:
    x = raw[RAW_COLS].copy()

    daily = raw["daily_screen_time_hours"].replace(0, np.nan)
    parts = raw[["social_media_hours", "gaming_hours", "work_study_hours"]].sum(axis=1)

    # Features with strong public validation in S6E8.
    x["parts_sum"] = parts
    x["social_media_share"] = raw["social_media_hours"] / daily
    x["gaming_share"] = raw["gaming_hours"] / daily
    x["work_study_share"] = raw["work_study_hours"] / daily
    x["weekend_minus_daily"] = raw["weekend_screen_time"] - raw["daily_screen_time_hours"]

    # Useful structural residual: how much reported daily screen time remains
    # after the three named components.
    x["daily_minus_parts"] = raw["daily_screen_time_hours"] - parts

    # A few low-risk continuous relationships; no explicit missingness flags.
    sleep = raw["sleep_hours"].replace(0, np.nan)
    opens = raw["app_opens_per_day"].replace(0, np.nan)
    x["screen_to_sleep"] = raw["daily_screen_time_hours"] / sleep
    x["notifications_per_open"] = raw["notifications_per_day"] / opens

    if encoded is not None:
        for c in encoded.columns:
            x[c] = encoded[c].to_numpy(dtype=np.float32)

    return x


def to_xgb_frame(x: pd.DataFrame) -> pd.DataFrame:
    x = x.copy()
    for c in CAT_COLS:
        if isinstance(x[c].dtype, pd.CategoricalDtype):
            x[c] = x[c].cat.codes.replace(-1, np.nan)
        else:
            x[c] = pd.Categorical(x[c]).codes
            x.loc[x[c] < 0, c] = np.nan
    for c in x.columns:
        x[c] = pd.to_numeric(x[c], errors="coerce")
    return x.astype(np.float32)


def to_cat_frame(raw: pd.DataFrame) -> pd.DataFrame:
    # CatBoost diversity view: raw + structural continuous features,
    # deliberately without target encodings.
    x = add_features(raw, encoded=None)
    for c in CAT_COLS:
        x[c] = x[c].astype("string").fillna("__MISSING__").astype(str)
    return x


def rank01(a: np.ndarray) -> np.ndarray:
    # AUC depends only on ordering. Percentile ranks make model scales comparable.
    return pd.Series(np.asarray(a)).rank(method="average").to_numpy(dtype=np.float64) / (len(a) + 1.0)


def fold_auc_vector(y: np.ndarray, pred: np.ndarray, fold_id: np.ndarray) -> np.ndarray:
    scores = []
    for f in np.unique(fold_id):
        m = fold_id == f
        scores.append(roc_auc_score(y[m], pred[m]))
    return np.asarray(scores, dtype=np.float64)


def choose_rank_blend(
    y: np.ndarray,
    preds: dict[str, np.ndarray],
    fold_id: np.ndarray,
) -> tuple[dict[str, float], np.ndarray]:
    names = list(preds)
    ranks = {k: rank01(v) for k, v in preds.items()}

    print("\nOOF base scores")
    for k in names:
        f = fold_auc_vector(y, ranks[k], fold_id)
        print(f"  {k:10s}: auc={roc_auc_score(y, ranks[k]):.7f}  folds={np.round(f, 7)}")

    # Coarse simplex grid is intentionally conservative to reduce blend-weight overfit.
    grid = np.arange(0.0, 1.0001, 0.05)
    best = None

    if len(names) == 3:
        for w0 in grid:
            for w1 in grid:
                w2 = 1.0 - w0 - w1
                if w2 < -1e-12:
                    continue
                w2 = max(0.0, w2)
                w = np.array([w0, w1, w2], dtype=float)
                blend = sum(w[i] * ranks[names[i]] for i in range(3))
                fs = fold_auc_vector(y, blend, fold_id)
                global_auc = roc_auc_score(y, blend)
                # Favor high global AUC while mildly penalizing fold instability.
                objective = global_auc - 0.15 * float(np.std(fs))
                candidate = (objective, global_auc, float(np.mean(fs)), -float(np.std(fs)), w, blend)
                if best is None or candidate[:4] > best[:4]:
                    best = candidate
    elif len(names) == 2:
        for w0 in grid:
            w = np.array([w0, 1.0 - w0], dtype=float)
            blend = w[0] * ranks[names[0]] + w[1] * ranks[names[1]]
            fs = fold_auc_vector(y, blend, fold_id)
            global_auc = roc_auc_score(y, blend)
            objective = global_auc - 0.15 * float(np.std(fs))
            candidate = (objective, global_auc, float(np.mean(fs)), -float(np.std(fs)), w, blend)
            if best is None or candidate[:4] > best[:4]:
                best = candidate
    else:
        raise ValueError("Need 2 or 3 model predictions for blending")

    assert best is not None
    weights = {names[i]: float(best[4][i]) for i in range(len(names))}
    print("\nSelected OOF rank blend")
    print("  weights:", weights)
    print(f"  global OOF AUC: {best[1]:.7f}")
    print("  fold AUCs:", np.round(fold_auc_vector(y, best[5], fold_id), 7))
    return weights, best[5]


## 4. Cross-validation, model training, blending, and submission

In [ ]:
def main() -> None:
    paths = locate_competition_files()
    print("Files:")
    print(" train :", paths.train)
    print(" test  :", paths.test)
    print(" sample:", paths.sample)

    train = pd.read_csv(paths.train)
    test = pd.read_csv(paths.test)
    sample = pd.read_csv(paths.sample)

    validate_raw(train, test, sample)
    train, test = prep_categories(train, test)

    y = train[TARGET].to_numpy(dtype=np.int8)
    gpu = has_gpu()
    print(f"\ntrain={train.shape}, test={test.shape}, positive_rate={y.mean():.6f}, GPU={gpu}")

    try:
        import lightgbm as lgb
    except Exception as e:
        raise RuntimeError("LightGBM is required in the Kaggle environment") from e

    try:
        import xgboost as xgb
    except Exception as e:
        raise RuntimeError("XGBoost is required in the Kaggle environment") from e

    try:
        from catboost import CatBoostClassifier
        cat_available = True
    except Exception:
        CatBoostClassifier = None
        cat_available = False
        print("CatBoost unavailable: continuing with LightGBM + XGBoost.")

    outer = StratifiedKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=SEED)

    oof_lgb = np.full(len(train), np.nan, dtype=np.float64)
    oof_xgb = np.full(len(train), np.nan, dtype=np.float64)
    oof_cat = np.full(len(train), np.nan, dtype=np.float64) if cat_available else None

    pred_lgb = np.zeros(len(test), dtype=np.float64)
    pred_xgb = np.zeros(len(test), dtype=np.float64)
    pred_cat = np.zeros(len(test), dtype=np.float64) if cat_available else None

    fold_id = np.full(len(train), -1, dtype=np.int8)

    for fold, (fit_idx, val_idx) in enumerate(outer.split(train, y), start=1):
        print(f"\n{'='*28} FOLD {fold}/{OUTER_FOLDS} {'='*28}")

        tr_raw = train.iloc[fit_idx].reset_index(drop=True)
        va_raw = train.iloc[val_idx].reset_index(drop=True)
        te_raw = test.reset_index(drop=True)
        y_tr = y[fit_idx]
        y_va = y[val_idx]
        fold_id[val_idx] = fold

        enc_tr, enc_va, enc_te = nested_exact_value_te(
            tr_raw, y_tr, va_raw, te_raw, seed=SEED + 101 * fold
        )

        xtr_lgb = add_features(tr_raw, enc_tr)
        xva_lgb = add_features(va_raw, enc_va)
        xte_lgb = add_features(te_raw, enc_te)

        # LightGBM: proven strong TE + ratio view.
        lgbm = lgb.LGBMClassifier(
            objective="binary",
            metric="auc",
            n_estimators=4500,
            learning_rate=0.03,
            num_leaves=31,
            max_depth=-1,
            min_child_samples=100,
            subsample=0.90,
            subsample_freq=1,
            colsample_bytree=0.90,
            reg_alpha=0.10,
            reg_lambda=2.0,
            max_bin=255,
            random_state=SEED + fold,
            n_jobs=-1,
            verbosity=-1,
        )

        lgbm.fit(
            xtr_lgb,
            y_tr,
            eval_set=[(xva_lgb, y_va)],
            eval_metric="auc",
            categorical_feature=CAT_COLS,
            callbacks=[lgb.early_stopping(180, verbose=False), lgb.log_evaluation(0)],
        )
        pva_lgb = lgbm.predict_proba(xva_lgb, num_iteration=lgbm.best_iteration_)[:, 1]
        pte_lgb = lgbm.predict_proba(xte_lgb, num_iteration=lgbm.best_iteration_)[:, 1]
        oof_lgb[val_idx] = pva_lgb
        pred_lgb += pte_lgb / OUTER_FOLDS
        print(f"LGB  fold AUC={roc_auc_score(y_va, pva_lgb):.7f}  best_iter={lgbm.best_iteration_}")

        # XGBoost: same leakage-safe signal, different tree geometry.
        xtr_xgb = to_xgb_frame(xtr_lgb)
        xva_xgb = to_xgb_frame(xva_lgb)
        xte_xgb = to_xgb_frame(xte_lgb)

        xgb_params = dict(
            objective="binary:logistic",
            eval_metric="auc",
            n_estimators=3500,
            learning_rate=0.035,
            max_depth=8,
            min_child_weight=20.0,
            subsample=0.90,
            colsample_bytree=0.90,
            reg_alpha=0.10,
            reg_lambda=2.0,
            max_bin=256,
            tree_method="hist",
            random_state=SEED + 1000 + fold,
            n_jobs=-1,
            early_stopping_rounds=180,
        )
        if gpu:
            # Works on current Kaggle XGBoost; fall back automatically if unsupported.
            xgb_params["device"] = "cuda"

        xgbm = xgb.XGBClassifier(**xgb_params)
        try:
            xgbm.fit(xtr_xgb, y_tr, eval_set=[(xva_xgb, y_va)], verbose=False)
        except Exception as e:
            if xgb_params.get("device") == "cuda":
                print("XGBoost CUDA path failed; retrying on CPU:", str(e)[:160])
                xgb_params.pop("device", None)
                xgbm = xgb.XGBClassifier(**xgb_params)
                xgbm.fit(xtr_xgb, y_tr, eval_set=[(xva_xgb, y_va)], verbose=False)
            else:
                raise

        pva_xgb = xgbm.predict_proba(xva_xgb)[:, 1]
        pte_xgb = xgbm.predict_proba(xte_xgb)[:, 1]
        oof_xgb[val_idx] = pva_xgb
        pred_xgb += pte_xgb / OUTER_FOLDS
        best_it = getattr(xgbm, "best_iteration", None)
        print(f"XGB  fold AUC={roc_auc_score(y_va, pva_xgb):.7f}  best_iter={best_it}")

        # CatBoost: lower standalone ceiling can still add useful non-tree-view diversity.
        if cat_available:
            xtr_cat = to_cat_frame(tr_raw)
            xva_cat = to_cat_frame(va_raw)
            xte_cat = to_cat_frame(te_raw)

            cat_params = dict(
                iterations=2600,
                learning_rate=0.035,
                depth=8,
                loss_function="Logloss",
                eval_metric="AUC",
                l2_leaf_reg=6.0,
                random_seed=SEED + 2000 + fold,
                verbose=False,
                allow_writing_files=False,
                random_strength=0.5,
                od_type="Iter",
                od_wait=180,
                thread_count=-1,
            )
            if gpu:
                cat_params.update(task_type="GPU", devices="0")

            catm = CatBoostClassifier(**cat_params)
            try:
                catm.fit(
                    xtr_cat,
                    y_tr,
                    cat_features=CAT_COLS,
                    eval_set=(xva_cat, y_va),
                    verbose=False,
                )
            except Exception as e:
                if cat_params.get("task_type") == "GPU":
                    print("CatBoost GPU path failed; retrying on CPU:", str(e)[:160])
                    cat_params.pop("task_type", None)
                    cat_params.pop("devices", None)
                    catm = CatBoostClassifier(**cat_params)
                    catm.fit(
                        xtr_cat,
                        y_tr,
                        cat_features=CAT_COLS,
                        eval_set=(xva_cat, y_va),
                        verbose=False,
                    )
                else:
                    raise

            pva_cat = catm.predict_proba(xva_cat)[:, 1]
            pte_cat = catm.predict_proba(xte_cat)[:, 1]
            oof_cat[val_idx] = pva_cat
            pred_cat += pte_cat / OUTER_FOLDS
            print(f"CAT  fold AUC={roc_auc_score(y_va, pva_cat):.7f}")

            del xtr_cat, xva_cat, xte_cat, catm, pva_cat, pte_cat

        del (
            tr_raw,
            va_raw,
            enc_tr,
            enc_va,
            enc_te,
            xtr_lgb,
            xva_lgb,
            xte_lgb,
            xtr_xgb,
            xva_xgb,
            xte_xgb,
            lgbm,
            xgbm,
            pva_lgb,
            pte_lgb,
            pva_xgb,
            pte_xgb,
        )
        gc.collect()

    if np.isnan(oof_lgb).any() or np.isnan(oof_xgb).any():
        raise RuntimeError("Incomplete OOF predictions")
    if cat_available and np.isnan(oof_cat).any():
        raise RuntimeError("Incomplete CatBoost OOF predictions")

    oof_models = {"lgb": oof_lgb, "xgb": oof_xgb}
    test_models = {"lgb": pred_lgb, "xgb": pred_xgb}
    if cat_available:
        oof_models["cat"] = oof_cat
        test_models["cat"] = pred_cat

    weights, blend_oof = choose_rank_blend(y, oof_models, fold_id)

    # Apply the exact same rank-space combination to test predictions.
    final_pred = np.zeros(len(test), dtype=np.float64)
    for name, weight in weights.items():
        final_pred += weight * rank01(test_models[name])

    # The rank blend is already in [0,1], but clipping protects against FP edge cases.
    final_pred = np.clip(final_pred, 1e-7, 1 - 1e-7)

    submission = sample.copy()
    submission[TARGET] = final_pred

    # Hard fail before writing a malformed Kaggle file.
    if len(submission) != len(test):
        raise RuntimeError("Submission row count mismatch")
    if not submission[ID_COL].equals(test[ID_COL].reset_index(drop=True)):
        raise RuntimeError("Submission ID order mismatch")
    if submission[TARGET].isna().any() or not np.isfinite(submission[TARGET]).all():
        raise RuntimeError("Submission contains NaN/inf")
    if not submission[TARGET].between(0, 1).all():
        raise RuntimeError("Submission probabilities outside [0,1]")

    submission.to_csv(paths.output, index=False)

    oof_out = pd.DataFrame(
        {
            ID_COL: train[ID_COL].to_numpy(),
            TARGET: y,
            "fold": fold_id,
            "lgb": oof_lgb,
            "xgb": oof_xgb,
            "blend": blend_oof,
        }
    )
    if cat_available:
        oof_out["cat"] = oof_cat
    oof_out.to_csv(paths.oof, index=False)

    print("\n" + "=" * 78)
    print("FINAL")
    print(f"OOF blend AUC : {roc_auc_score(y, blend_oof):.7f}")
    print("Blend weights :", weights)
    print("Rows          :", len(submission))
    print("Prediction min:", float(submission[TARGET].min()))
    print("Prediction mean:", float(submission[TARGET].mean()))
    print("Prediction max:", float(submission[TARGET].max()))
    print("Saved         :", paths.output)
    print("OOF saved     :", paths.oof)
    print("=" * 78)


## 5. Run the complete pipeline

In [ ]:
if __name__ == "__main__":
    main()
